# Sample diffusion checkpoints

This notebook follows `transformer/sample.ipynb`: sample the all-geometry models, compare the fine-tuned and scratch models at $h^{1,2}=5$, and sample conditioned on $N_{\rm flux}$, $|W_0|$, or $1/g_s$. It supports all 12 checkpoints in `diffusion/checkpoints/`.

**Choose one setup route:** run **A. Local Jupyter** on your computer, or **B. Google Colab** in Colab. Skip the other route, then continue at **C. Shared sampling setup**. Run the optional quick sampling check before starting larger runs. Do not use “Run All” across both setup routes.

## A. Local Jupyter setup

Run this block in your local notebook kernel and skip all of section B. It locates your existing checkout from the current directory or its parents, and saves samples in `diffusion/samples`. It does not clone the repository, install packages, or mount Google Drive.

Select a kernel with the project dependencies installed. If needed, install them from the repository root into that kernel's environment using `python -m pip install -r requirements-cpu.txt` for CPU, or `python -m pip install -r requirements-gpu.txt` for a compatible CUDA environment. Restart the kernel after installation. If checkpoint files have not been downloaded, run `git lfs pull` from the repository root.

Leave `REPO_ROOT=None` when Jupyter starts in the repository root or `diffusion/`. Otherwise replace it with `Path("/absolute/path/to/towards_type_IIB_foundation_model")`. The block prints the Python executable so you can verify which environment Jupyter is using. CPU sampling is supported; the shared settings select CUDA when available.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = None  # Or Path("/absolute/path/to/towards_type_IIB_foundation_model").
if REPO_ROOT is None:
    REPO_ROOT = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / "diffusion/src/pipeline.py").is_file()),
        None,
    )
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT to your local repository path above.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "diffusion/src/pipeline.py").is_file():
    raise FileNotFoundError(f"No diffusion/src/pipeline.py in {REPO_ROOT}")
SAMPLE_DIR = REPO_ROOT / "diffusion/samples"
print(f"Python: {sys.executable}")
print(f"Repository: {REPO_ROOT}")
print(f"Local sample directory: {SAMPLE_DIR}")

## B. Google Colab setup

**Skip this entire section when using local Jupyter.** In Colab, select a GPU runtime, then run the installation block once. It clones the repository with Git LFS and installs the project dependencies. Restart the kernel if requested, then run the following Colab paths and Drive block.

In [ ]:
%cd /content
!apt-get install git-lfs
!git lfs install
!git clone https://github.com/StringJAX/towards_type_IIB_foundation_model.git
%cd towards_type_IIB_foundation_model
%pip install -r requirements.txt
%cd diffusion

### Colab paths and Google Drive

Run this block after installation, including after a kernel restart. It uses the Colab checkout at `/content/towards_type_IIB_foundation_model`, mounts Google Drive, and saves samples in `MyDrive/IIB_foundation_model_samples`. Continue with section C afterward.

In [ ]:
from pathlib import Path
from google.colab import drive

REPO_ROOT = Path("/content/towards_type_IIB_foundation_model")
if not (REPO_ROOT / "diffusion/src/pipeline.py").is_file():
    raise FileNotFoundError("Run the Colab installation block first.")
drive.mount("/content/drive")
SAMPLE_DIR = Path("/content/drive/MyDrive/IIB_foundation_model_samples")
print(f"Repository: {REPO_ROOT}")
print(f"Drive sample directory: {SAMPLE_DIR}")

## C. Shared sampling setup

Run this section after completing either A or B. The next block makes the selected checkout importable, creates the sample directory, and lists the supported checkpoint filenames. The sampling helper accepts any listed filename or an explicit checkpoint path.

Each model example below draws **100 samples per geometry** by default. The optional quick check draws only two samples for each of the five $h^{1,2}=5$ geometries.

In [ ]:
import sys

if "REPO_ROOT" not in globals() or "SAMPLE_DIR" not in globals():
    raise RuntimeError("Run either the local Jupyter setup or the Colab setup first.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = REPO_ROOT / "diffusion/checkpoints"
CHECKPOINTS = sorted(path.name for path in CHECKPOINT_DIR.glob("*.pt"))
if not CHECKPOINTS:
    raise FileNotFoundError(f"No supported checkpoints found in {CHECKPOINT_DIR}; download them with git lfs pull.")
print("Available checkpoints:\n" + "\n".join(CHECKPOINTS))
print(f"Samples will be saved in {SAMPLE_DIR}")

## Import the sampling helper and set shared options

**Rerun this block after updating the repository code.** It reloads the sample-format module and sampling pipeline before importing `generate`, so an existing kernel uses the current exporter. The printed source path and sample format identify the code being used. If another checkout is already imported, the block asks you to restart the kernel.

The next block imports `generate` from `diffusion.src.pipeline` and collects the shared settings in `SAMPLING_OPTIONS`. Each example passes these options to the imported function. Rerun this block after changing a shared setting. Change `checkpoint` to select the pre-trained or GRPO version and `file_name` to choose the output name. The mode is inferred from the checkpoint's model configuration. Use `h12=5` for the fine-tuned and scratch comparisons; `h12=None` samples all 20 geometries in the supplied checkpoints.

- `NUM_SAMPLES` is the number of samples **per geometry**. Lower `BATCH_SIZE` if GPU memory is limited. Reproducibility requires the same seed, device, batch size and sampling settings.
- `generate` uses confidence-order sampling by default. Together with `TEMPERATURE=0.4`, this matches the evaluation recipe in `../diffusion-transformer/`. The conditional examples explicitly use guidance 2.0; the geometry-only examples use 1.0. Set temperature or guidance to `None` to use checkpoint settings (falling back to 1.0 when absent). The training mask schedule and time conditioning always come from the checkpoint.
- Each call writes a `.p` file containing NumPy dictionaries with integer `flux` and complex-valued `z` and `tau` arrays in physical units and a `.json` sidecar recording the selected geometries, target, seed and sampler settings. Keep both files together. Existing files are protected; choose a new `file_name` when rerunning.
- Initializers are saved without rounding or discretization. Both parts of the predicted axio-dilaton are retained in every mode. Older integer-encoded sample files must be regenerated with a new filename before using the new evaluator.
- Keep `ONLY_SAMPLE=True` to evaluate later on a CPU runtime. Set it to `False` to run physics evaluation immediately after saving, in a separate CPU process within this runtime. This still occupies the GPU runtime while evaluation runs. Evaluation checks each sample on its prompted geometry.

In [ ]:
import importlib
import torch

# Refresh dependencies before rebinding functions in a long-lived notebook kernel.
importlib.invalidate_caches()
for module_name in (
    "diffusion.src.evaluation.samples",
    "diffusion.src.evaluation",
    "diffusion.src.pipeline",
):
    module = importlib.import_module(module_name)
    if not Path(module.__file__).resolve().is_relative_to(REPO_ROOT):
        raise RuntimeError(f"{module_name} was imported from {module.__file__}. "
                           "Restart the kernel and select the intended repository.")
    importlib.reload(module)
from diffusion.src.pipeline import generate
from diffusion.src.evaluation.samples import SAMPLE_FORMAT
print(f"Sampler: {module.__file__}; sample format: {SAMPLE_FORMAT}")

NUM_SAMPLES = 100
BATCH_SIZE = 64
SEED = 90261001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEMPERATURE = 0.4
ONLY_SAMPLE = True
RESULTS_DIR = SAMPLE_DIR / "evaluations"
print(f"Device: {DEVICE}; default samples per geometry: {NUM_SAMPLES}")

SAMPLING_OPTIONS = {
    "sample_dir": SAMPLE_DIR,
    "results_dir": RESULTS_DIR,
    "num_samples": NUM_SAMPLES,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "device": DEVICE,
    "temperature": TEMPERATURE,
    "only_sample": ONLY_SAMPLE,
}


## 1. Sample from the pre-trained and GRPO refined model

These models are conditioned on geometry, without an additional observable target. This block samples all 20 geometries, analogous to transformer `RUN_ID=37`.

Choose `"uncon_pretrained.pt"` for the pre-trained model or `"uncon_grpo.pt"` for its GRPO refinement. The filename determines the default output name, so the two versions can be saved separately. Set `file_name` to a string to choose your own name. All shared settings above apply.

In [ ]:
checkpoint = "uncon_grpo.pt"  # Or "uncon_pretrained.pt".
file_name = None  # For example, "all_h12_sample".
unconditional_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name)

## 2. Sample from the fine-tuned model

This is the counterpart of transformer `RUN_ID=45`: a model transferred from lower-$h^{1,2}$ geometries and fine-tuned at $h^{1,2}=5$. Only the five $h^{1,2}=5$ geometries are sampled, even though the checkpoints contain metadata for all 20 geometries.

Choose `"h12_5_finetuned_pretrain.pt"` for the fine-tuned model before GRPO or `"h12_5_finetuned_grpo.pt"` for its GRPO refinement. Keep `h12=5` and use the same sample count, seed, and sampler settings for this comparison.


In [ ]:
checkpoint = "h12_5_finetuned_grpo.pt" # Or "h12_5_finetuned_pretrain.pt"
file_name = None
finetuned_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name, h12=5)

### Scratch benchmark at $h^{1,2}=5$

This block is the counterpart of transformer `RUN_ID=44`. It samples the same five geometries from the model trained from scratch, providing the benchmark for the fine-tuned model.

Choose `"h12_5_scratch_pretrain.pt"` or `"h12_5_scratch_grpo.pt"`. Use the same sample count, seed and sampler settings as the fine-tuned example for a controlled comparison.

In [ ]:
checkpoint = "h12_5_scratch_grpo.pt"  # Or "h12_5_scratch_pretrain.pt".
file_name = None
scratch_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name, h12=5)

## 3. Sample conditioned on $N_{\rm flux}$, $|W_0|$ or $1/g_s$

The following blocks are analogous to the three conditional examples in `transformer/sample.ipynb`. They sample all 20 geometries with one observable target in addition to the geometry. The targets below match that notebook's examples; change them to study another target. The requested target is saved in the JSON sidecar and read back by `diffusion/src/evaluation/evaluate.py`.

### $N_{\rm flux}$

Choose `"nflux_pretrained.pt"` or `"nflux_grpo.pt"`, analogous to transformer `RUN_ID=38`. `nflux` must be an integer. The example requests $N_{\rm flux}=40$. Guidance 2.0 controls the conditional sampling strength.

In [ ]:
checkpoint = "nflux_grpo.pt"  # Or "nflux_pretrained.pt".
file_name = None
nflux_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name, nflux=40,
                      guidance_scale=2.0)

### $|W_0|$

Choose `"w0_pretrained.pt"` or `"w0_grpo.pt"`, analogous to transformer `RUN_ID=39`. Set `w0` to a finite, non-negative physical target. This example requests $|W_0|=1.1$.

In [ ]:
checkpoint = "w0_grpo.pt"  # Or "w0_pretrained.pt".
file_name = None
w0_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name, w0=1.1,
                   guidance_scale=2.0)

### $1/g_s$

Choose `"gs_pretrained.pt"` or `"gs_grpo.pt"`, analogous to transformer `RUN_ID=46`. Set `gs_inv` to a finite, positive target for $1/g_s$. This example requests $1/g_s=4.0$.

The diffusion conditioner uses $g_s$, so the helper passes `1 / gs_inv` to the model. The full complex predicted axio-dilaton is saved and used as the solver initializer; its imaginary part is not replaced by the requested target. Any finite positive target is supported, without a discretization grid.

In [ ]:
checkpoint = "gs_grpo.pt"  # Or "gs_pretrained.pt".
file_name = None
gs_inv_file = generate(checkpoint, **SAMPLING_OPTIONS, file_name=file_name, gs_inv=4.0,
                       guidance_scale=2.0)

## Evaluate saved samples later

The `.p` files can be evaluated without loading a neural-network checkpoint. Keep each `.json` sidecar beside its `.p` file: it records the physical sample format, geometry panel, and observable target. Older integer-encoded files must be regenerated; the evaluator rejects them instead of interpreting their integers as physical coordinates.

Open `diffusion/evaluate.ipynb`, choose its local or Colab setup, and set `SAMPLES_DIR` to the directory containing your samples. The notebook imports `evaluate_samples` from `diffusion.src.evaluation.evaluate`. Evaluation uses JAXVacua and SciPy directly, checks each sample on its prompted geometry, and reports validity and unique-valid counts.

Command-line examples from the repository root:

```bash
python -m diffusion.src.evaluation.evaluate --samples_file diffusion_h12_5_finetuned_grpo_samples.p
python -m diffusion.src.evaluation.evaluate --samples_file diffusion_gs_grpo_samples.p
```

For a Drive file, pass its absolute path to `--samples_file`. Outputs are `*_valid.p` and `*_summary.json`. Observable prompt keys and target values are expressed in physical units.
